# 00. Progressive Disclosure：能力按需加载

把 100 个能力全部写进 system prompt？100K token 起步，且稀释注意力。Agent Skills 的解法是三层披露：

| 层 | 内容 | 何时加载 | 典型成本 |
|----|------|---------|---------|
| 1. Metadata | name + description（frontmatter） | 常驻 | ~100 token/skill |
| 2. Instructions | SKILL.md 正文 | skill 被触发时 | ~1-2K token |
| 3. Resources | 正文引用的脚本/参考文件 | 实际用到时 | 按需 |

关键洞察：Layer 1 让 agent「知道自己会什么」，Layer 2 才教它「怎么做」——和人类的「查目录 vs 读章节」一样。

> **检查点**：能算一笔账：50 个 skill，全量注入 vs 三层加载（平均触发 2 个），各花多少 token？

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()

🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 解剖本仓库的真实 skill：.agents/skills/benches-development
from pathlib import Path
import re

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

skill_md = ROOT / '.agents' / 'skills' / 'benches-development' / 'SKILL.md'
text = skill_md.read_text()

fm = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)$", text, re.DOTALL)
metadata, body = fm.group(1), fm.group(2)

print(f"Skill: {skill_md.parent.name}")
print(f"Layer 1 (metadata): {len(metadata)} chars ≈ {len(metadata)//4} tokens — 常驻")
print(f"Layer 2 (body):     {len(body)} chars ≈ {len(body)//4} tokens — 触发时加载")
print(f"\n--- metadata 原文 ---\n{metadata}")
print(f"--- body 前 200 字符 ---\n{body[:200]}...")

Skill: benches-development
Layer 1 (metadata): 153 chars ≈ 38 tokens — 常驻
Layer 2 (body):     4145 chars ≈ 1036 tokens — 触发时加载

--- metadata 原文 ---
name: benches-development
description: Use when modifying this benches repository, adding experiments, tests, benchmarks, tooling, or dependency scripts.
--- body 前 200 字符 ---
# Benches Development Rules

## Repository Boundaries

- `backups/` is source material only. Do not modify, delete, reformat, or migrate from it unless the user explicitly asks.
- `libs/` contains reu...


## 设计 SKILL.md 的原则

- **description 是触发器**：写清「什么时候用我」，比写清「我是什么」更重要
- **正文是指令不是文档**：祈使句、规则列表、边界条件；散文式说明浪费 token
- **资源外置**：长参考表、脚本放单独文件，正文只写「需要时读 xxx」
- 反面教材：description 写「一个很有用的工具」— 匹配器无从下手

In [3]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'agent-skills' / 'scripts' / 'skill_loader_demo.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--query', '把 100 华氏度换算成摄氏度', '--use-api', '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)

=== Layer 1: metadata resident (1 skills) ===
  - unit-converter: 单位换算专家。当用户要求在不同度量单位之间转换（长度、重量、温度、数据量等）时使用。始终展示换算公式与系数。
  常驻成本 ≈ 18 tokens（vs 全量注入见 Layer 2）

=== Selection (LLM) ===
  → unit-converter

=== Layer 2: body loaded on trigger (≈106 tokens) ===
# Unit Converter Skill

执行单位换算时遵守以下规则：

1. **先写公式**：`结果 = 输入 × 系数`，再代入数字。
2. **系数表**：
   - 长度：1 km = 0.621371 mile；1 mile = 1.60934 km；1 m = 3.28084 ft
   - 重量：1 kg = 2.20462 lb；1 lb = 0.453592 kg
   - 温度：°F = °C × 9/5 + 32；°C = (°F − 32) × 5/9
   - 数据量：1 GiB = 1024 MiB；1 GB = 1000 MB（注意二进制/十进制差异，必须注明用的是哪个）
3. **精度**：保留 4 位有效数字，注明舍入。
4. **双向给出**：给出换算结果后，附一行反向验证（结果换回输入）。
5. 数据量换算必须显式区分 GiB（1024 进制）...

=== Answer comparison ===
[无 skill]
100 华氏度等于约 37.78 摄氏度。计算公式：\( (100 - 32) \times \frac{5}{9} = 68 \times \frac{5}{9} \approx 37.78 \)。

[注入 skill]
100 华氏度换算为摄氏度：

**公式**：°C = (°F − 32) × 5/9  
**计算**：°C = (100 − 32) × 5/9 = 68 × 5/9 = 340/

Takeaway: skill = 按需注入的指令包。metadata 常驻让 agent「知道会什么」，
body 触发时加载让 agent「知道怎么做」— 全量注入既贵又稀释注意力。



## 真实触发观察

注意上面两次回答的差异：无 skill 时模型自由发挥；注入 skill 后严格按规则（公式 → 系数 → 精度 → 反向验证）。
**这就是「能力工程化」：把最佳实践固化成可复用、可版本管理、可测试的指令包。**

## 练习

1. 给 `agents/agent-skills/skills/` 添加一个你领域的 SKILL.md（如 sql-optimizer），用 loader 验证触发率：写 5 个该触发的问题 + 2 个不该触发的，统计准确率。
2. 统计 `.agents/skills/` 下所有 skill 的 Layer 1 总 token 成本；如果团队有 200 个 skill，metadata 常驻还成立吗？（引出：skill 也需要检索/RAG）
3. 阅读本仓库 `benches-development` skill 正文，找出三条「祈使句规则」，改写一条散文句为规则句。